# Week 6 Lab. AR, MA & ARMA
**Time Series Analysis & Random Processes · 30-minute voluntary practice**

먼저 **파일 → Drive에 사본 저장**을 누르세요. CPU로 실행합니다. 외부 데이터 다운로드는 없습니다.

| Section | Do / 할 일 | Matching slides (titles) / 짝 슬라이드 제목 |
|---|---|---|
| 1 · 8 min | Predict how one extra shock spreads / 충격의 영향 예측 | 「One Extra Shock Can Affect Many Later Values」 · 「A Negative φ Makes the Effect Flip Signs」 · 「An MA(1) Shock Ends After One Extra Step」 · 「Can a Stable Model Look Like It Has a Trend?」 |
| 2 · 9 min | Compare theoretical and sample ACF/PACF / 이론과 표본 비교 | 「For AR(1), ACF Fades Gradually but PACF Stops After Lag 1」 · 「For MA(1), ACF Stops After Lag 1 but PACF Fades Gradually」 · 「For ARMA, Both ACF and PACF Can Fade Gradually」 · 「Will a Sample Plot Have Exact Zeros?」 · 「More Observations Make the Pattern Easier to See」 |
| 3 · 8 min | Fit candidates and inspect residuals / 적합과 잔차 확인 | 「We Know the True Parameters. How Close Does Fitting Get?」 · 「Why Not Just Add More Parameters?」 · 「Did the Model Leave a Predictable Pattern Behind?」 |
| 4 · 5 min | Score the forecasts on held-out data / 보류 자료로 예측 평가 | 「Did the Fitted Model Beat a Simple Forecast?」 |

**위에서 아래로 실행하세요.** 빈칸은 4개이며 각각 한 줄입니다. TODO 뒤 Self-check를 먼저 실행한 다음, 필요할 때 접힌 정답을 열어 해당 줄을 복사하세요. 정답은 Markdown에만 있어 **Run all이 내 답을 덮어쓰지 않습니다.** 빈칸이 남아 있으면 안내 오류가 나며, 정상입니다. 네 빈칸을 채우면 전체 실행이 가능합니다.

학습 내용: 한 충격의 효과, 표본 ACF의 흔들림, ARMA 적합과 잔차, 보류 자료의 예측 평가. 과제나 제출물이 아닙니다. Colab 기본 패키지가 없으면 아래 안내에 따라 설치한 뒤 Setup을 다시 실행하세요.

## 0. Setup / 준비
그대로 실행하세요. 여기서 정의하는 도구의 구현은 암기 대상이 아닙니다.
모든 모형은 독립인 평균 0, 분산 1의 정규 충격을 사용합니다. `burnin=500`은 초기값 0의 영향을 충분히 줄이기 위해 앞부분을 버리는 설정입니다. 이는 무한 과거에서 시작하는 정상 과정을 유한하게 근사합니다.

`ARIMA(p, 0, q)`의 0은 차분을 하지 않는다는 뜻입니다. 이번 시간은 이미 정상인 모의자료를 사용합니다. 글꼴 문제를 피하기 위해 그림 라벨은 영문입니다.

In [ ]:
import sys, importlib
required = ['numpy', 'pandas', 'matplotlib', 'statsmodels']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError('패키지가 없습니다. 새 셀에 %pip install ' + ' '.join(missing) + ' 를 실행한 뒤 Setup을 다시 실행하세요.')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.arima.model import ARIMA
SEED = 20261006
COLORS = ['#1E4A32', '#D9AE3A', '#B34F3F']
plt.rcParams.update({'figure.figsize': (10, 3.2), 'font.size': 11})

def simulate(kind, n=400, seed=SEED):
    # The minus sign in the AR coefficient array is a library convention.
    specs = {'AR': ([1, -0.7], [1]), 'MA': ([1], [1, 0.7]),
             'ARMA': ([1, -0.6], [1, 0.5]), 'AR2': ([1, -1.2, 0.5], [1]),
             'AR95': ([1, -0.95], [1])}
    ar, ma = specs[kind]
    rng = np.random.default_rng(seed)
    return ArmaProcess(ar, ma).generate_sample(nsample=n, burnin=500,
                                               distrvs=rng.standard_normal)

def correlations(x, max_lag=12):
    return (acf(x, nlags=max_lag, fft=False)[1:],
            pacf(x, nlags=max_lag, method='ywm')[1:])

def bars(ax, values, title, band=None):
    ax.bar(np.arange(1, len(values) + 1), values, color=COLORS[0], width=0.45)
    ax.axhline(0, color='gray', lw=0.7)
    if band is not None:
        ax.axhline(band, color=COLORS[1], ls='--')
        ax.axhline(-band, color=COLORS[1], ls='--')
    ax.set(xlabel='lag', ylabel='correlation', title=title, ylim=(-0.6, 1))

print('Setup ready / 준비 완료')
print('Python:', sys.version.split()[0], 'NumPy:', np.__version__, 'statsmodels:', statsmodels.__version__)

## 1. How long does one extra shock last? / 한 충격이 얼마나 남을까?
짝 슬라이드 「One Extra Shock Can Affect Many Later Values」 · 「A Negative φ Makes the Effect Flip Signs」 · 「An MA(1) Shock Ends After One Extra Step」. 두 경로에서 **오늘 충격 하나만 +1**로 바꾸고 나머지는 같게 둡니다. 그 차이를 그립니다. 임의의 한 경로 자체를 그리는 실험과 다릅니다.

AR(1)의 모형 가정은 $x_t=\phi x_{t-1}+\varepsilon_t$. 따라서 두 경로의 차이는 한 단계마다 $\phi$배가 됩니다. MA(1)은 $x_t=\varepsilon_t+\theta\varepsilon_{t-1}$라서 효과가 `1, θ, 0, 0, …`입니다. 우선 `φ=0.7`, `φ=-0.7`, `φ=1`의 차이를 말로 예상해보세요.

In [ ]:
# TODO 1: one AR update / AR 한 걸음
def ar_step(previous, shock, phi):
    """Instructions: return the new value using the slide 「In AR(1), Today Depends on Yesterday Plus a New Shock」.
    Args: previous, shock, phi: scalar floats, shape ().
    """
    ###################################################################
    raise NotImplementedError('정상입니다. 빈칸에 AR 한 걸음을 작성하세요. / Fill the one-line AR update.')
    ###################################################################

In [ ]:
# Self-check BEFORE opening the solution / 정답을 보기 전에 실행
for previous, shock, phi, expected in [(2, -0.2, 0.7, 1.2), (2, 0, -0.7, -1.4), (3, 0.5, 0, 0.5)]:
    assert np.isclose(ar_step(previous, shock, phi), expected), '이전 값에 phi를 곱한 뒤 새 shock을 더하세요.'
print('TODO 1 passed / 통과')

<details><summary>Solution 1 / 필요할 때만 펼치기</summary>

빈칸의 `raise` 한 줄을 아래 줄로 바꾸세요.
```python
return phi * previous + shock
```
</details>

In [ ]:
# TODO 2: one MA update / MA 한 걸음
def ma_step(shock_now, shock_previous, theta):
    """Instructions: return the value built from two shocks (slide 「MA(1) Combines New and Previous Shocks」).
    Args: shock_now, shock_previous, theta: scalar floats, shape ().
    """
    ###################################################################
    raise NotImplementedError('정상입니다. 빈칸에 MA 한 걸음을 작성하세요. / Fill the one-line MA update.')
    ###################################################################

In [ ]:
for now, previous, theta, expected in [(-0.2, 1, 0.7, 0.5), (0.5, 2, -0.5, -0.5), (1, 4, 0, 1)]:
    assert np.isclose(ma_step(now, previous, theta), expected), '현재 shock + theta × 이전 shock을 계산하세요.'
print('TODO 2 passed / 통과')

<details><summary>Solution 2 / 필요할 때만 펼치기</summary>

```python
return shock_now + theta * shock_previous
```
</details>

In [ ]:
def ar_response(phi, horizon=12, extra=1.0):
    effect = [extra]
    for _ in range(horizon):
        effect.append(ar_step(effect[-1], 0.0, phi))
    return np.asarray(effect)

def ma_response(theta, horizon=12, extra=1.0):
    shock = np.zeros(horizon + 2)
    shock[1] = extra
    return np.array([ma_step(shock[h + 1], shock[h], theta) for h in range(horizon + 1)])

horizon = np.arange(13)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.7))
for phi, color in [(0.7, COLORS[0]), (1.0, COLORS[1])]:
    axes[0].plot(horizon, ar_response(phi), '.-', color=color, label=f'AR phi={phi}')
for phi, color in [(0.7, COLORS[0]), (-0.7, COLORS[1])]:
    axes[1].plot(horizon, ar_response(phi), '.-', color=color, label=f'AR phi={phi}')
axes[2].plot(horizon, ar_response(0.7), '.-', color=COLORS[0], label='AR phi=0.7')
axes[2].plot(horizon, ma_response(0.7), '.-', color=COLORS[1], label='MA theta=0.7')
for ax in axes:
    ax.set(xlabel='steps after extra shock', ylabel='effect')
    ax.axhline(0, color='gray', lw=0.6)
    ax.legend()
plt.tight_layout(); plt.show()
print('AR(0.7), h=0..4:', np.round(ar_response(0.7)[:5], 4))
print('MA(0.7), h=0..4:', ma_response(0.7)[:5])

**What to notice.** AR keeps passing the effect through past values. A negative coefficient changes its sign at each step. MA keeps the effect for a fixed number of steps.
> 요약: AR은 점차 감쇠하고, MA(1)은 한 단계 뒤까지만 영향을 남깁니다. 예상 출력은 AR `[1, .7, .49, .343, .2401]`, MA `[1, .7, 0, 0, 0]`입니다.

**Try it.** 마지막 출력에서 `extra=2`를 사용해보세요. 음의 φ와 φ=1도 비교하세요. φ=1의 경로는 정상 AR 모형으로 해석하지 않습니다.

### A stable model that looks like a trend / 추세처럼 보이는 정상 모형
짝 슬라이드 「Can a Stable Model Look Like It Has a Trend?」. 같은 충격으로 `φ=0.95` 와 `φ=0.7` 의 경로를 160걸음 그립니다. 둘 다 |φ| < 1 인 정상 모형이고 장기 평균은 0 입니다. **예상**: 0.95 쪽이 0 으로 돌아오는 데 얼마나 걸릴까요? 짧은 구간만 보면 추세처럼 보일까요?

In [ ]:
# Same shocks, two carry-over values / 같은 충격, 다른 φ. Seed SEED + 4 is the one on the slide.
fig, ax = plt.subplots(figsize=(10, 3.4))
for kind, phi, color in [('AR95', 0.95, COLORS[0]), ('AR', 0.7, COLORS[1])]:
    path = simulate(kind, n=160, seed=SEED + 4)
    ax.plot(np.arange(160), path, color=color, label=f'AR(1), phi={phi}')
    print(f'phi={phi}: first three {np.round(path[:3], 3)}, last three {np.round(path[-3:], 3)}, mean {path.mean():.2f}')
ax.axhline(0, color='gray', lw=0.6)
ax.set(xlabel='time step', ylabel='deviation'); ax.legend(); plt.show()

**What to notice.** Both paths share the same shocks. With φ = 0.95 a shock loses only 5% per step, so the path wanders away from 0 for long stretches, and the last 60 steps look like an upward trend. A short rising stretch cannot tell a slowly fading shock from one that stays. Week 7 adds a test for this.
> 요약: 같은 충격인데 0.95 는 한 걸음에 5% 만 잊어 오래 떠돕니다. 짧은 상승 구간만으로는 「천천히 사라지는 충격」과 「남는 충격(랜덤워크)」을 구분할 수 없습니다. 7주차의 단위근 검정이 이 질문을 다룹니다.

**Try it.** `0.95` 를 `0.3` 으로 바꿔 보세요(Setup 의 `specs` 에 `'AR30': ([1, -0.3], [1])` 를 더하면 됩니다). `φ=1` 은 정상 모형이 아니므로 여기서는 비교하지 않습니다.

## 2. Theory versus a sample / 이론과 표본
짝 슬라이드 「For AR(1), ACF Fades Gradually but PACF Stops After Lag 1」 · 「For MA(1), ACF Stops After Lag 1 but PACF Fades Gradually」 · 「For ARMA, Both ACF and PACF Can Fade Gradually」 · 「Will a Sample Plot Have Exact Zeros?」 · 「More Observations Make the Pattern Easier to See」. ACF는 떨어진 두 값의 상관, PACF는 중간 값들의 선형 영향을 제거한 뒤 남는 상관입니다. AR(1)은 이론적으로 PACF가 lag 1 뒤에서 0이고, MA(1)은 ACF가 lag 1 뒤에서 0입니다. **표본의 막대는 정확히 0일 필요가 없습니다.**

이론값은 모형 계수로 계산합니다. 표본값은 400개 관측값으로 추정합니다. 두 값을 같은 것으로 읽지 마세요.

**ARMA 의 표본 그림(n = 400)은 강의의 mystery series 와 같은 계열입니다**(「Can We Forecast This Series Better Than a Simple Guess?」 · 「Back to the Mystery Series. Does It Look Like AR(1)?」 · 「Mystery Series Again. Which Model Should We Try?」).

In [ ]:
specs = {'AR': ([1, -0.7], [1]), 'MA': ([1], [1, 0.7]),
         'ARMA': ([1, -0.6], [1, 0.5])}
for kind, (ar, ma) in specs.items():
    process = ArmaProcess(ar, ma)
    sample = simulate(kind)
    sample_a, sample_p = correlations(sample)
    fig, axes = plt.subplots(2, 2, figsize=(10, 6))
    bars(axes[0, 0], process.acf(lags=13)[1:], f'{kind}: theoretical ACF')
    bars(axes[0, 1], process.pacf(lags=13)[1:], f'{kind}: theoretical PACF')
    bars(axes[1, 0], sample_a, f'{kind}: sample ACF, n=400')
    bars(axes[1, 1], sample_p, f'{kind}: sample PACF, n=400')
    plt.tight_layout(); plt.show()
print('MA theoretical lag-1 ACF:', round(0.7 / (1 + 0.7**2), 3))

### What changes with sample size? / 표본이 많아지면?
슬라이드 「More Observations Make the Pattern Easier to See」과 같은 난수 시드와 모형을 사용합니다. `±2/√n`은 **white noise를 가정한 각 lag의 근사 기준**입니다. 일반 ARMA의 정확한 신뢰구간이나 모든 lag를 동시에 덮는 밴드가 아닙니다.
처음에는 슬라이드와 같은 막대만 보고, 다음 셀의 `show_band=True`로 바꿔 기준선도 확인하세요.

In [ ]:
show_band = False
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, n_obs in zip(axes, [50, 400]):
    sample = simulate('AR', n=n_obs, seed=SEED + 7)
    sample_a, _ = correlations(sample)
    band = 2 / np.sqrt(n_obs)
    bars(ax, sample_a, f'AR(1), n={n_obs}', band=band if show_band else None)
    print(f'n={n_obs}: white-noise reference band ±{band:.3f}')
plt.tight_layout(); plt.show()

**What to notice.** A theoretical cutoff is an exact statement about a model. Sample bars fluctuate, especially with fewer observations. A stray bar does not determine the model order.
> 요약: 이론상 0인 값도 표본에서는 흔들립니다. 밴드 밖 막대 하나를 보고 모형을 확정하지 마세요.

**Try it.** 시드를 `SEED+8`로 바꿔 두 표본을 다시 비교하세요. 표본이 많아지면 모든 개별 막대가 항상 더 정확해진다는 보장은 없습니다.

## 3. Fit and inspect residuals / 적합과 잔차 확인
짝 슬라이드 「We Know the True Parameters. How Close Does Fitting Get?」 · 「Why Not Just Add More Parameters?」 · 「Did the Model Leave a Predictable Pattern Behind?」. 강의의 mystery series 가 바로 이 계열입니다. 생성 모형은 ARMA(1,1), φ=0.6, θ=0.5입니다. 500개를 만들고 **앞의 400개만** 모든 후보 적합에 사용합니다. 나머지 100개는 §4 의 예측 평가에만 쓰고 이번 AIC/BIC 비교에는 사용하지 않습니다.

잔차는 `관측값 − 한 단계 앞 예측값`입니다. 잔차에 여러 lag의 패턴이 남으면 모형이 의존을 놓쳤을 수 있습니다. AIC와 BIC는 적합도와 복잡도를 함께 고려하는 제공 지표로, **동일 자료에 적합한 후보끼리 낮은 값을 선호**합니다. 이 값만으로 미래 예측 성능을 보장하지는 않습니다.

In [ ]:
# TODO 3: translate ARMA(p,q) to this library's ARIMA order
def arma_order(p, q):
    """Instructions: use zero differences for this stationary data (slide 「How Does the Computer Fit an ARMA Model?」).
    Args: p, q: nonnegative Python int scalars. Returns: tuple of 3 ints.
    """
    ###################################################################
    raise NotImplementedError('정상입니다. 차분 횟수를 0으로 둔 순서 튜플을 쓰세요. / Return the order tuple.')
    ###################################################################

In [ ]:
for p, q, expected in [(1, 0, (1, 0, 0)), (0, 1, (0, 0, 1)), (1, 1, (1, 0, 1)), (2, 1, (2, 0, 1))]:
    assert arma_order(p, q) == expected, 'ARIMA 순서는 (p, 차분 횟수, q)입니다.'
print('TODO 3 passed / 통과')

<details><summary>Solution 3 / 필요할 때만 펼치기</summary>

```python
return (p, 0, q)
```
</details>

In [ ]:
values = simulate('ARMA', n=500)
train, holdout = values[:400], values[400:]
orders = {'WN': (0, 0), 'AR(1)': (1, 0), 'MA(1)': (0, 1), 'ARMA(1,1)': (1, 1)}
fits = {}
rows = []
for name, (p, q) in orders.items():
    fitted = ARIMA(train, order=arma_order(p, q), trend='n').fit()
    fits[name] = fitted
    rows.append({'Model': name, 'AIC': fitted.aic, 'BIC': fitted.bic,
                 'Converged': fitted.mle_retvals.get('converged', False)})
comparison = pd.DataFrame(rows).set_index('Model')
print(comparison.round(1).to_string())
fitted = fits['ARMA(1,1)']
print('\nFitted coefficients / 추정 계수:')
print(pd.Series(fitted.params, index=fitted.param_names).round(3).to_string())
assert comparison.Converged.all(), '일부 적합이 수렴하지 않았습니다. 그 후보의 점수는 아직 신뢰하지 마세요.'
assert np.isfinite(comparison[['AIC', 'BIC']].to_numpy()).all(), '적합 점수가 유한한지 확인하세요.'
print('Held out for later evaluation / 평가용 보류:', len(holdout))

기본 시드에서 대략 φ=0.630, θ=0.465, shock variance=1.012가 나옵니다. AIC는 WN 1581.0, AR(1) 1198.4, MA(1) 1272.2, ARMA(1,1) 1147.3입니다. 패키지 버전이나 최적화 설정에 따라 마지막 자리에는 차이가 날 수 있습니다.

`Non-invertible starting MA parameters ... Using zeros` 경고는 **초기 추정값을 바꿨다는 안내**입니다. 최종 적합 실패와 같지 않으며 `Converged`를 확인하세요. `ConvergenceWarning`이 뜨거나 수렴이 False이면 결과 해석 전에 문제를 해결해야 합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, name in zip(axes, ['WN', 'ARMA(1,1)']):
    residual_a = acf(fits[name].resid, nlags=12, fft=False)[1:]
    bars(ax, residual_a, f'{name}: residual ACF')
plt.tight_layout(); plt.show()

**What to notice.** The ARMA fit uses the simulated dependence and leaves much less visible linear correlation. A quiet residual ACF does not prove independence or future forecasting skill.
> 요약: AIC/BIC로 후보를 비교한 뒤 잔차를 확인합니다. 실제 활용에는 시간 순서를 지킨 예측평가도 필요합니다.

**Try it.** 생성 모형을 `AR` 또는 `MA`로 바꿔 같은 후보를 비교하세요. 표본마다 언제나 진짜 모형이 선택된다는 보장은 없습니다. §4 에서 보류한 100개로 단순 기준 예측과 비교합니다.

## 4. Did the fitted model beat a simple forecast? / 보류 자료로 예측 평가
짝 슬라이드 「Did the Fitted Model Beat a Simple Forecast?」. §3 에서 보류한 마지막 100개(401–500)를 처음 씁니다. 계수는 처음 400개로 맞춘 것을 그대로 두고, 매 시점 **그 직전까지의 값만으로** 다음 값을 예측합니다(one step ahead). 모형이 없는 기준 둘과 비교합니다: 「마지막 값 그대로」와 「0, 알려진 평균」. 백색잡음 후보의 예측이 곧 0 예측입니다.

**예상**: ARMA(1,1) 이 「마지막 값」 예측을 이길까요? t = 400 에서 100개를 한꺼번에 예측하면 어느 쪽이 이길까요?

RMSE (root mean squared error) 는 예측 오차를 제곱해 평균한 뒤 제곱근을 취한 값입니다. 자료와 같은 단위이고, 같은 구간에서는 작을수록 좋습니다.

In [ ]:
# TODO 4: root mean squared error / 오차 제곱 평균의 제곱근
def rmse(actual, forecast):
    """Instructions: square each error, average, then take the square root (slide 「Did the Fitted Model Beat a Simple Forecast?」).
    Args: actual, forecast: 1-D arrays of the same length. Returns: a scalar float.
    """
    ###################################################################
    raise NotImplementedError('정상입니다. 빈칸에 RMSE 한 줄을 작성하세요. / Fill the one-line RMSE.')
    ###################################################################

In [ ]:
# Self-check BEFORE opening the solution / 정답을 보기 전에 실행
for actual, forecast, expected in [([1, 2], [1, 4], np.sqrt(2)), ([3, 3, 3], [3, 3, 3], 0.0), ([0, 0, 0, 0], [2, 2, 2, 2], 2.0)]:
    assert np.isclose(rmse(np.array(actual), np.array(forecast)), expected), '오차를 제곱해 평균한 뒤 제곱근을 취하세요.'
print('TODO 4 passed / 통과')

<details><summary>Solution 4 / 필요할 때만 펼치기</summary>

```python
return np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2))
```
</details>

In [ ]:
def one_step_forecasts(fitted, full_series, start):
    # Same coefficients as the fit, applied to all 500 values. Each forecast uses only the values before that time step.
    return fitted.apply(full_series).get_prediction(start=start, end=len(full_series) - 1).predicted_mean

rows = []
for name in fits:
    rows.append({'Forecast': name,
                 'One step ahead': rmse(holdout, one_step_forecasts(fits[name], values, len(train))),
                 'All 100 made at t=400': rmse(holdout, fits[name].forecast(len(holdout)))})
last_value = np.r_[train[-1], holdout[:-1]]          # yesterday's value as today's forecast / 어제 값을 오늘 예측으로
rows.append({'Forecast': 'Last observed value',
             'One step ahead': rmse(holdout, last_value),
             'All 100 made at t=400': rmse(holdout, np.full(len(holdout), train[-1]))})
scores = pd.DataFrame(rows).set_index('Forecast')
print('Holdout RMSE, 100 values / 보류 100개의 RMSE (units)')
print(scores.round(3).to_string())

**What to notice.** One step ahead, ARMA(1,1) scores best, and the fitted-model ranking happens to match AIC in this simulation; that is not guaranteed. Forecast all 100 values at once from t = 400 and every score is about the same: for these stable short-memory models, long-horizon forecasts converge toward the mean, so their fixed-origin long-horizon scores become similar.
> 요약: 한 걸음 앞에서는 ARMA(1,1) 이 가장 작고, 그 순서가 이 시뮬레이션에서는 AIC 순서와 같습니다(보장은 아닙니다). 기본 시드의 기대 출력은 ARMA(1,1) 1.052, AR(1) 1.120, 마지막 값 1.172, MA(1) 1.279, WN(0 예측) 1.915 입니다. t = 400 에서 100개를 한꺼번에 예측하면 전부 약 1.92 입니다. 이 정상 · 짧은 기억 모형들은 먼 지평의 예측이 평균 0 으로 수렴하므로 고정 원점의 먼 지평 점수가 서로 비슷해지고, 모형의 값어치는 짧은 지평에서 납니다. 보류 구간 하나의 숫자이므로 소수 둘째 자리 차이에 뜻을 두지 마세요.

**Try it.** 생성 모형을 `AR` 로 바꿔 §3 부터 다시 실행하면 어느 후보가 한 걸음 앞에서 이기나요? 이 결과를 보고 모형을 다시 고르면, 그 100개는 더 이상 순수한 평가 구간이 아닙니다.

## 5. One observation in your own words / 한 문장 해석
- 바꾼 설정:
- 예상한 변화:
- 실제 관찰:
- 모형의 어떤 규칙으로 설명할 수 있나:

제출하지 않습니다. 개인 오류는 코드 셀 번호와 마지막 오류 메시지를 함께 수업 게시판에 남기세요.

References: [statsmodels ARIMA](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html), [FPP3 §9.5](https://otexts.com/fpp3/non-seasonal-arima.html).